## Demo notebook

In this demo we will go through the basic functionalities of `RotOptSynth`.

In [1]:
import numpy as np
from scipy.stats import unitary_group
import rotoptsynth as ros
import pennylane as qml

In [2]:
# Pick some system size and generate a target with that size.
n = 4
n = 7
N = 2**n
wires = range(n)
dim = 4**n
U = unitary_group.rvs(N, random_state=81512)

In [4]:
# Perform unitary synthesis with validation enabled.
ros.disable_validation()
%prun -s tottime ops = ros.rot_opt_qsd(U, wires=wires, zeroed_wires=None)

# Additionally check manually that the matrix is reproduced correctly:
print(f"Matrix reproduced correctly: {np.allclose(ros.ops_to_mat(ops, wires), U)}")

 Matrix reproduced correctly: True


         7751956 function calls (7658292 primitive calls) in 1.982 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
273949/236063    0.137    0.000    0.535    0.000 autoray.py:30(do)
    97543    0.068    0.000    0.164    0.000 wires.py:42(_process)
679975/679967    0.054    0.000    0.081    0.000 {built-in method builtins.isinstance}
   174384    0.053    0.000    0.085    0.000 interface_utils.py:151(_get_interface_of_single_tensor)
    39732    0.046    0.000    0.195    0.000 operation.py:1099(__init__)
    50646    0.045    0.000    0.054    0.000 wires.py:467(<genexpr>)
    32762    0.044    0.000    0.243    0.000 utils.py:228(cast)
   150818    0.043    0.000    0.132    0.000 interface_utils.py:87(get_interface)
   281104    0.039    0.000    0.100    0.000 autoray.py:379(_choose_backend)
   346628    0.036    0.000    0.074    0.000 autoray.py:527(get_lib_fn)
631095/582484    0.035    0.000    0.040    0.000 {bu

In [5]:
# Count the number of rotation angles in the operators and compare to the group dimension
num_rotation_angles = ros.count_rotation_angles(ops)
print(f"Number of rotation angles ({num_rotation_angles}) matches group dimension ({dim}): {num_rotation_angles==dim}")

Number of rotation angles (16384) matches group dimension (16384): True


In [8]:
# Draw the circuit
print(qml.drawer.tape_text(qml.tape.QuantumScript(ops), show_matrices=False, max_length=100, wire_order=wires))

0: ────────────────────────────────╭GlobalPhase────────────────────────────────────────────── ···
1: ────────────────────────────────├GlobalPhase────────────────────────────────────────────── ···
2: ────────────────────────────────├GlobalPhase────────────────────────────────────────────── ···
3: ────────────────────────────────├GlobalPhase────────────────────────────────────────────── ···
4: ────────────────────────────────├GlobalPhase─╭RZ(M4)───────────────────────────────╭RY(M5) ···
5: ──U(M0)─╭X──RZ─╭●─────╭X──U(M2)─├GlobalPhase─├◑───────RY──RZ─╭●──RX─╭●──RZ──RY──RZ─├◑───── ···
6: ──U(M1)─╰●──RY─╰X──RY─╰●──U(M3)─╰GlobalPhase─╰◑───────RY──RZ─╰X──RZ─╰X──RZ──RY──RZ─╰◑───── ···

0: ··· ─────────────────────────────────────────────────────────────────────────────────── ···
1: ··· ─────────────────────────────────────────────────────────────────────────────────── ···
2: ··· ─────────────────────────────────────────────────────────────────────────────────── ···
3: ··· ─────────────────────

In [9]:
# Count gates:
@qml.specs
@qml.qnode(qml.device("default.qubit"))
def node():
    for op in ops:
        qml.apply(op)

    return qml.expval(qml.Z(0))
    
resources = node()["resources"]
gate_counts = dict(resources.gate_types)
gate_counts

{'QubitUnitary': 4,
 'CNOT': 127,
 'RZ': 8,
 'RY': 6,
 'GlobalPhase': 1,
 'SelectPauliRot': 851,
 'RX': 1}

In [10]:
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(ros.asymmetric_two_qubit_decomp, show_matrices=False)(U, wires=wires))

0: ───────────╭●──U(M0)─╭●──RX(1.36)─╭●──U(M2)─╭GlobalPhase(-0.52)─┤  
1: ──RZ(2.00)─╰X──U(M1)─╰X──RZ(0.97)─╰X──U(M3)─╰GlobalPhase(-0.52)─┤  


In [14]:
from rotoptsynth.flag_decomp import _flag_decomp_two_qubits
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(_flag_decomp_two_qubits, show_matrices=False)(U, wires=wires))

0: ─╭U(M0)──RY(0.92)──RZ(1.63)─╭●──RX(1.31)─╭●──RZ(9.72)──RY(0.77)──RZ(1.15)─┤  
1: ─╰U(M0)──RY(1.14)──RZ(0.61)─╰X──RZ(0.81)─╰X──RZ(1.69)──RY(0.71)──RZ(3.99)─┤  


In [15]:
ros.attach_multiplexer_node([qml.RY(0.6, 0)], [qml.RY(-0.1, 0)], "mpx")

[SelectPauliRot(array([ 0.6, -0.1]), wires=['mpx', 0])]

In [16]:
ops0 = [qml.SelectPauliRot([0.6, 0.7], [0], target_wire="target", rot_axis="X")]
ops1 = [qml.SelectPauliRot([0.2, -0.5], [0], target_wire="target", rot_axis="X")]
ros.attach_multiplexer_node(ops0, ops1, "new mpx")

[SelectPauliRot(array([ 0.6,  0.7,  0.2, -0.5]), wires=['new mpx', 0, 'target'])]

In [17]:
ros.attach_multiplexer_node([qml.CNOT([0, 1])], [qml.CNOT([0, 1])], "mpx")

[CNOT(wires=[0, 1])]